In [ ]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

In [ ]:
# TOP 10 NGÀY CÓ NHIỆT ĐỘ DẦU TRUNG BÌNH CAO NHẤT KHI MÁY VẬN HÀNH
q4_high_temp_days = spark.sql('''
    SELECT
        date,
        ROUND(AVG(Oil_temperature), 2) AS nhiet_do_dau_tb,
        ROUND(MAX(Oil_temperature), 2) AS nhiet_do_dau_cao_nhat,
        ROUND(AVG(Motor_current), 2) AS dong_dien_tb,
        COUNT(*) AS so_ban_ghi
    FROM sensor
    WHERE COMP = 0
    GROUP BY date
    ORDER BY nhiet_do_dau_tb DESC
    LIMIT 10
''')
# Hiển thị Execution Plan
print("\n========== EXECUTION PLAN ==========")
q4_high_temp_days.explain(True)
print("\n--- TOP 10 NGÀY CÓ NHIỆT ĐỘ DẦU TRUNG BÌNH CAO NHẤT ---")
df_q4 = q4_high_temp_days.toPandas()
try:
    display(df_q4)
except NameError:
    print(df_q4.to_string(index=False))